# Downsample to 3k Training Pool
Stratified sample 3000 from `fever_train_joined.jsonl`, then filter `sft_data.jsonl` and `dpo_data.jsonl` to those IDs.
Original 5k files are untouched.

In [7]:
import json
import random
from collections import Counter, defaultdict

random.seed(42)
TRAIN_SIZE = 3000

## Step 1: Sample 3k from fever_train_joined (restricted to IDs in sft_data)

In [8]:
# get IDs that already have teacher-generated sft data (no new API calls needed)
sft_ids = set()
with open("data/generated/sft_data.jsonl") as f:
    for line in f:
        sft_ids.add(json.loads(line)["id"])
print(f"sft_data IDs: {len(sft_ids)}")

# load fever_train_joined, keep only IDs covered by sft_data
all_train = []
with open("data/joined/fever_train_joined.jsonl") as f:
    for line in f:
        ex = json.loads(line)
        if ex["id"] in sft_ids:
            all_train.append(ex)
print(f"fever_train_joined (filtered to sft coverage): {len(all_train)}")
print(Counter(ex["label"] for ex in all_train))

sft_data IDs: 4999
fever_train_joined (filtered to sft coverage): 4999
Counter({'NOT MENTIONED': 1667, 'SUPPORTED': 1667, 'CONTRADICTED': 1665})


In [9]:
# stratified sample: equal per class
buckets = defaultdict(list)
for ex in all_train:
    buckets[ex["label"]].append(ex)

per_class = TRAIN_SIZE // len(buckets)
train_pool_3k = []
for lbl, items in sorted(buckets.items()):
    random.shuffle(items)
    train_pool_3k.extend(items[:per_class])
    print(f"  {lbl}: {len(items)} available, took {per_class}")

random.shuffle(train_pool_3k)
pool_ids = {ex["id"] for ex in train_pool_3k}

POOL_OUT = "data/generated/train_pool_3k.jsonl"
with open(POOL_OUT, "w") as f:
    for ex in train_pool_3k:
        f.write(json.dumps(ex) + "\n")
print(f"\ntrain_pool_3k: {len(train_pool_3k)} saved to {POOL_OUT}")
print(Counter(ex["label"] for ex in train_pool_3k))

  CONTRADICTED: 1665 available, took 1000
  NOT MENTIONED: 1667 available, took 1000
  SUPPORTED: 1667 available, took 1000

train_pool_3k: 3000 saved to data/generated/train_pool_3k.jsonl
Counter({'NOT MENTIONED': 1000, 'SUPPORTED': 1000, 'CONTRADICTED': 1000})


## Step 2: Filter sft_data to 3k pool IDs

In [10]:
sft_3k = []
with open("data/generated/sft_data.jsonl") as f:
    for line in f:
        ex = json.loads(line)
        if ex["id"] in pool_ids:
            sft_3k.append(ex)

SFT_OUT = "data/generated/sft_data_3k.jsonl"
with open(SFT_OUT, "w") as f:
    for ex in sft_3k:
        f.write(json.dumps(ex) + "\n")
print(f"sft_data_3k: {len(sft_3k)} saved to {SFT_OUT}")
print(Counter(ex["label"] for ex in sft_3k))

sft_data_3k: 3000 saved to data/generated/sft_data_3k.jsonl
Counter({'NOT MENTIONED': 1000, 'CONTRADICTED': 1000, 'SUPPORTED': 1000})
